# AutoPilot Studio — Watch Motion Smoke Test

**Purpose:** create one real 3-second AI-generated moving clip of mechanical watch gears. This is a motion-pipeline test, not the finished 18–20 second ad.

## How to run
1. Open this notebook in Google Colab.
2. Choose **Runtime → Change runtime type → T4 GPU** if available. Free GPU allocation is not guaranteed.
3. Tap **Runtime → Run all** and wait for the model download and generation to finish.
4. The final cell previews and downloads `autopilot_watch_motion_test.mp4`.

The notebook uses the official Wan-Video/Wan2.1 code and the 1.3B text-to-video checkpoint. It downloads several GB and may take a while. Do not enable paid services or accept any upgrade prompt. If no GPU is offered, stop rather than pay.

**Quality boundary:** this first test is 832×480 landscape. It is intended to verify genuine object motion before spending time on more clips. It is not yet a vertical, brand-ready, client-cleared ad.

In [ ]:
import os, subprocess, sys

# Check the hosted runtime before downloading model weights or installing packages.
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU is attached. In Colab choose Runtime → Change runtime type → GPU, "
        "then rerun. Free GPU access is not guaranteed; do not pay to continue."
    )
print("GPU runtime detected.\n", gpu_check.stdout[:2500])

# Install the normal inference dependencies while keeping the runtime's existing PyTorch/CUDA intact.
# flash-attn is intentionally skipped because compiling it often fails in hosted notebooks.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.31.0", "transformers>=4.49.0", "tokenizers>=0.20.3",
    "accelerate>=1.1.1", "opencv-python>=4.9.0.80", "tqdm", "imageio",
    "easydict", "ftfy", "imageio-ffmpeg", "huggingface_hub", "numpy<2"
], check=True)

# Import torch only after pip has settled NumPy and other dependencies.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("NVIDIA GPU was detected by nvidia-smi but PyTorch cannot access CUDA. Stop here; do not pay for compute.")
print("PyTorch CUDA device:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

repo_dir = "/content/Wan2.1"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Wan-Video/Wan2.1.git", repo_dir], check=True)
print("Dependencies and official inference code are ready.")

In [ ]:
from huggingface_hub import snapshot_download
model_dir = "/content/Wan2.1-T2V-1.3B"
print("Downloading the open Wan2.1 T2V-1.3B checkpoint; this is several GB...")
snapshot_download(repo_id="Wan-AI/Wan2.1-T2V-1.3B", local_dir=model_dir)
print("Model files downloaded to:", model_dir)

In [ ]:
prompt = (
    "A premium fictional unbranded mechanical wristwatch movement in an obsidian-black studio, "
    "extreme macro cinematography. Several tiny champagne-gold gears rotate continuously at different "
    "speeds while a polished balance wheel oscillates rhythmically; two small steel components rise, "
    "rotate and precisely align with the gear train. A narrow strip of warm light sweeps across brushed "
    "metal and sapphire crystal. Slow controlled camera orbit and subtle dolly-in, physically plausible "
    "mechanics, realistic reflections, crisp metal detail, premium luxury-product lighting, one uninterrupted "
    "shot, clearly visible continuous mechanical motion throughout, centered composition, no words, no logo."
)
negative = (
    "static image, still picture, slideshow, blurry, low quality, flicker, jitter, melting metal, "
    "random gears, duplicated watch, malformed dial, fake text, logo, watermark, overexposed highlights, "
    "crushed black detail, unstable camera, sudden zoom"
)
output_path = "/content/autopilot_watch_motion_test.mp4"
cmd = [
    sys.executable, "generate.py",
    "--task", "t2v-1.3B",
    "--size", "832*480",
    "--ckpt_dir", model_dir,
    "--offload_model", "True",
    "--t5_cpu",
    "--sample_shift", "8",
    "--sample_guide_scale", "6",
    "--frame_num", "49",
    "--save_file", output_path,
    "--prompt", prompt + " Negative prompt: " + negative
]
print("Generating one short moving clip. Runtime depends on the free GPU assigned.")
subprocess.run(cmd, cwd=repo_dir, check=True)
if not os.path.isfile(output_path) or os.path.getsize(output_path) < 100_000:
    raise RuntimeError("Output file is missing or unexpectedly small; do not mark the test successful.")
print("Generated file:", output_path, "— bytes:", os.path.getsize(output_path))

In [ ]:
from IPython.display import Video, display
import subprocess, json

probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_name,width,height,r_frame_rate",
    "-of", "json", output_path
], capture_output=True, text=True, check=True)
print(json.dumps(json.loads(probe.stdout), indent=2))
display(Video(output_path, embed=True))

# Download the real MP4 to the device. Review the full clip before using it anywhere.
from google.colab import files
files.download(output_path)

## Review gate
Pass this smoke test only if the video itself contains obvious gear/part movement—not just camera zoom—and the main object does not melt or flicker badly. If it fails, record the failure and adjust only the prompt or generation settings before retrying. The 480p landscape clip is a technical motion test, not a polished vertical deliverable.

The official Wan2.1 repository identifies the project as Apache-2.0 and documents the 1.3B model for 480p text-to-video. You remain responsible for respecting the model license, applicable law, and any third-party rights in the generated result.